# **SESIÓN 3:** Autocorrelación y Fundamentos de Pronóstico
## Universidad Autónoma de Occidente
### Maestría en Inteligencia Artificial y Ciencias de Datos
**Instructor:** Dr. Ing. Sergio A. Cantillo · sacantillo@uao.edu.co

---

### 🎯 Objetivos de la Sesión
- Detectar y tratar **problemas de calidad de datos** antes del modelado
- Comprender y aplicar la **ACF** como herramienta de diagnóstico
- Aplicar el **Test de Ljung-Box** para verificar estructura temporal
- Implementar y comparar **4 modelos baseline** con `statsforecast`
- Evaluar pronósticos con métricas formales usando `utilsforecast`
- Ejecutar **Time Series Cross-Validation** temporal

### 📦 Dataset
| Dataset | Área | Descripción |
|---------|------|-------------|
| **Perrin Frères — Champagne** | 🍾 Retail | Ventas mensuales 1964–1972 (108 obs.) |

---

## ⚙️ PARTE 1: Configuración del Entorno

In [1]:
!pip install statsforecast utilsforecast statsmodels -q

In [2]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, WindowAverage, RandomWalkWithDrift
from utilsforecast.losses import mae, rmse, smape
from utilsforecast.evaluation import evaluate
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("✅ Todas las librerías cargadas correctamente")

/home/alejo/.virtualenvs/datascience-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Todas las librerías cargadas correctamente


---
## 📦 PARTE 2: Carga de Datos y Formato Nixtla

El dataset de **Perrin Frères** contiene ventas mensuales de champagne en Francia (1964–1972).
Es un ejemplo clásico de serie con estacionalidad anual marcada (pico en diciembre) y ligera tendencia.

Lo cargamos directamente y lo convertimos al formato estándar `unique_id | ds | y`.


In [3]:
# ── Cargar dataset ──────────────────────────────────────────────────────
url = 'https://lazyprogrammer.me/course_files/timeseries/perrin-freres-monthly-champagne.csv'
try:
    raw = pd.read_csv(url, parse_dates=True, skipfooter=2, engine='python')
    ventas_series = raw.iloc[:, 0].dropna()
    print("✅ Descargado desde URL")
except Exception:
    # Datos embebidos como fallback (misma fuente, dataset canónico)
    vals = [2815,2672,2755,2721,2946,3036,2282,2212,2922,4301,5764,7132,
            2541,2475,3031,2764,2601,2648,2329,2363,2960,4813,6237,6136,
            2028,1920,2499,2808,2941,3143,3084,2513,3164,4425,5034,5534,
            2302,2049,2540,2945,2521,2751,3225,2611,3222,4842,5937,6800,
            2099,1998,2688,2936,2885,3020,2645,2745,3521,4630,5678,5897,
            1945,2046,2318,2685,2731,3010,2524,2630,3235,4413,5208,6087,
            1701,1715,2360,2560,2838,2640,2502,2513,3049,3989,4993,5500,
            2197,1927,2445,2784,2678,2736,2528,2671,3235,4452,5725,6040,
            2394,2155,2998,2568,2409,2731,2110,2784,3466,3898,5140,6034]
    dates = pd.date_range('1964-01', periods=108, freq='ME')
    ventas_series = pd.Series(vals, index=dates, name='Ventas')
    print("✅ Datos cargados desde fuente embebida")

# ── Convertir al formato Nixtla: unique_id | ds | y ──────────────────────
df = pd.DataFrame({
    'unique_id': 'Champagne',
    'ds':        ventas_series.index,
    'y':         ventas_series.values.astype(float)
})

print(f"\n📊 Dataset: {len(df)} observaciones mensuales")
print(f"   Período: {df.ds.min().strftime('%Y-%m')} → {df.ds.max().strftime('%Y-%m')}")
print(f"   Ventas  mín: {df.y.min():,.0f} | máx: {df.y.max():,.0f} | media: {df.y.mean():,.0f}")
df.head(6)

✅ Datos cargados desde fuente embebida

📊 Dataset: 108 observaciones mensuales
   Período: 1964-01 → 1972-12
   Ventas  mín: 1,701 | máx: 7,132 | media: 3,309


,unique_id,ds,y
0,Champagne,1964-01-31,2815.0
1,Champagne,1964-02-29,2672.0
2,Champagne,1964-03-31,2755.0
3,Champagne,1964-04-30,2721.0
4,Champagne,1964-05-31,2946.0
5,Champagne,1964-06-30,3036.0


In [4]:
# Visualización de la serie completa
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines+markers',
                          name='Ventas Champagne',
                          line=dict(color='#2196F3', width=1.8),
                          marker=dict(size=3)))
fig.update_layout(
    title='Ventas Mensuales de Champagne — Perrin Frères (1964–1972)',
    xaxis_title='Fecha', yaxis_title='Ventas (miles de botellas)',
    height=380, template='plotly_white'
)
fig.show()

### 🔍 ¿Qué observar?
- **Estacionalidad anual marcada:** cada diciembre hay un pico pronunciado (demanda navideña)
- **Tendencia leve:** el nivel promedio crece ligeramente a lo largo del período
- **Heteroscedasticidad:** las fluctuaciones estacionales parecen crecer con el nivel → candidato para log-transformación


---
## 🔍 PARTE 3: Calidad de Datos

Antes de cualquier análisis o modelado, hay que garantizar la calidad de los datos.
Problemas sin detectar aquí se propagan a todos los modelos y métricas.

Trabajamos con tres tipos de problemas:

| Problema | Síntoma | Estrategia |
|----------|---------|-----------|
| **Valores faltantes** | NaN o huecos temporales | Forward fill, interpolación, media estacional |
| **Outliers** | Puntos extremos atípicos | IQR, Z-score, Hampel |
| **Cambios de régimen** | Saltos estructurales abruptos | Inspección visual, prueba de Chow |


### 3.1 Valores Faltantes — Detección y Estrategias de Imputación

In [5]:
# ── Detectar valores faltantes ──────────────────────────────────────────
print("=" * 55)
print("  DIAGNÓSTICO DE VALORES FALTANTES")
print("=" * 55)

# Verificar continuidad temporal
expected_dates = pd.date_range(df.ds.min(), df.ds.max(), freq='ME')
missing_dates  = expected_dates.difference(df.ds)

print(f"  Observaciones esperadas: {len(expected_dates)}")
print(f"  Observaciones presentes: {len(df)}")
print(f"  Fechas faltantes:        {len(missing_dates)}")
print(f"  NaN en columna y:        {df.y.isna().sum()}")

if len(missing_dates) == 0 and df.y.isna().sum() == 0:
    print("\n  ✅ Serie completa — sin valores faltantes")
else:
    print(f"\n  ⚠️  Fechas faltantes: {missing_dates.tolist()}")

  DIAGNÓSTICO DE VALORES FALTANTES
  Observaciones esperadas: 108
  Observaciones presentes: 108
  Fechas faltantes:        0
  NaN en columna y:        0

  ✅ Serie completa — sin valores faltantes


In [6]:
# ── Simulación pedagógica: introducir 5 valores faltantes artificiales ──
import copy

df_con_nans = df.copy()
np.random.seed(42)
idx_faltantes = np.random.choice(range(10, len(df)-2), size=5, replace=False)
df_con_nans.loc[df_con_nans.index[idx_faltantes], 'y'] = np.nan

print(f"NaN introducidos: {df_con_nans.y.isna().sum()} en posiciones:")
print(df_con_nans[df_con_nans.y.isna()][['ds', 'y']])

# ── Tres estrategias de imputación ───────────────────────────────────────
df_ffill = df_con_nans.copy()
df_ffill['y'] = df_ffill['y'].ffill()                              # Forward fill

df_interp = df_con_nans.copy()
df_interp['y'] = df_interp['y'].interpolate(method='linear')       # Interpolación lineal

df_seas = df_con_nans.copy()                                        # Media estacional
for idx in df_con_nans[df_con_nans.y.isna()].index:
    mes = df_con_nans.loc[idx, 'ds'].month
    media_mes = df.loc[df.ds.dt.month == mes, 'y'].mean()
    df_seas.loc[idx, 'y'] = media_mes

# ── Comparación visual ────────────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines',
                          name='Original', line=dict(color='gray', width=1.5, dash='dot')))
fig.add_trace(go.Scatter(x=df_ffill.ds, y=df_ffill.y, mode='lines',
                          name='Forward Fill', line=dict(color='#2196F3', width=1.5)))
fig.add_trace(go.Scatter(x=df_interp.ds, y=df_interp.y, mode='lines',
                          name='Interpolación lineal', line=dict(color='#FF5722', width=1.5)))
fig.add_trace(go.Scatter(x=df_seas.ds, y=df_seas.y, mode='lines',
                          name='Media estacional', line=dict(color='#4CAF50', width=1.5)))

# Marcar posiciones de NaN
nan_ds = df_con_nans.loc[df_con_nans.y.isna(), 'ds']
fig.add_trace(go.Scatter(x=nan_ds, y=[df.y.max()*1.05]*len(nan_ds),
                          mode='markers', name='NaN introducidos',
                          marker=dict(color='red', size=10, symbol='x')))

fig.update_layout(title='Comparación de Estrategias de Imputación',
                  xaxis_title='Fecha', yaxis_title='Ventas',
                  height=420, template='plotly_white')
fig.show()

NaN introducidos: 5 en posiciones:
            ds   y
43  1967-08-31 NaN
83  1970-12-31 NaN
87  1971-04-30 NaN
90  1971-07-31 NaN
104 1972-09-30 NaN


### 🔍 Interpretación — Imputación
- **Forward Fill:** rápido y simple, pero propaga el último valor observado (puede distorsionar picos)
- **Interpolación lineal:** suaviza la transición entre valores conocidos — buena opción para huecos cortos
- **Media estacional:** usa el promedio histórico del mismo mes — la más robusta cuando hay estacionalidad fuerte

> **Regla práctica:** para series con estacionalidad clara como esta, la media estacional es preferida porque preserva el patrón anual.

> ⚠️ **Orden de operaciones:** la imputación se hace **antes** de los tests de estacionaridad y transformaciones (recordar Sesión 2).


### 3.2 Detección de Outliers — IQR y Z-score

In [7]:
# ── Método 1: IQR (Rango Intercuartílico) ────────────────────────────────
Q1, Q3 = df.y.quantile(0.25), df.y.quantile(0.75)
IQR    = Q3 - Q1
limite_inf_iqr = Q1 - 1.5 * IQR
limite_sup_iqr = Q3 + 1.5 * IQR
outliers_iqr = df[(df.y < limite_inf_iqr) | (df.y > limite_sup_iqr)]

print("IQR — Detección de Outliers")
print(f"  Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}")
print(f"  Límite inferior: {limite_inf_iqr:.0f}")
print(f"  Límite superior: {limite_sup_iqr:.0f}")
print(f"  Outliers detectados: {len(outliers_iqr)}")
print(outliers_iqr[['ds','y']])

# ── Método 2: Z-score ─────────────────────────────────────────────────────
z_scores = np.abs(stats.zscore(df.y))
outliers_z = df[z_scores > 3]
print(f"\nZ-score (|z| > 3) — Outliers detectados: {len(outliers_z)}")
if len(outliers_z): print(outliers_z[['ds','y']])

# ── Visualización comparativa ─────────────────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines',
                          name='Serie', line=dict(color='#2196F3', width=1.5)))
fig.add_hline(y=limite_sup_iqr, line_dash='dash', line_color='orange',
              annotation_text='Límite IQR superior')
fig.add_hline(y=limite_inf_iqr, line_dash='dash', line_color='orange',
              annotation_text='Límite IQR inferior')
if len(outliers_iqr):
    fig.add_trace(go.Scatter(x=outliers_iqr.ds, y=outliers_iqr.y,
                              mode='markers', name='Outlier (IQR)',
                              marker=dict(color='red', size=10, symbol='circle-open', line_width=2)))
fig.update_layout(title='Detección de Outliers — Método IQR',
                  height=380, template='plotly_white',
                  xaxis_title='Fecha', yaxis_title='Ventas')
fig.show()

IQR — Detección de Outliers
  Q1=2513  Q3=3615  IQR=1102
  Límite inferior: 860
  Límite superior: 5269
  Outliers detectados: 14
            ds       y
10  1964-11-30  5764.0
11  1964-12-31  7132.0
22  1965-11-30  6237.0
23  1965-12-31  6136.0
35  1966-12-31  5534.0
46  1967-11-30  5937.0
47  1967-12-31  6800.0
58  1968-11-30  5678.0
59  1968-12-31  5897.0
71  1969-12-31  6087.0
83  1970-12-31  5500.0
94  1971-11-30  5725.0
95  1971-12-31  6040.0
107 1972-12-31  6034.0

Z-score (|z| > 3) — Outliers detectados: 0


### 🔍 Interpretación — Outliers
- **IQR:** detecta puntos que se alejan más de 1.5×IQR del rango central. Robusto, no asume distribución normal.
- **Z-score:** detecta puntos a más de 3 desviaciones estándar. Sensible a la escala y asume normalidad aproximada.

En esta serie, los picos de diciembre son **outliers legítimos** (fenómeno estacional, no errores).
Hay que distinguir entre outliers *aditivos* (error de medición) y *innovacionales* (evento real):
- **Error de medición** → imputar o corregir
- **Evento real** → conservar, posiblemente modelar con variables dummy

> **Regla práctica:** aplicar IQR **dentro de cada mes** (por estacionalidad) evita falsos positivos en series estacionales.


In [8]:
# ── IQR por mes: más apropiado para series estacionales ─────────────────
df_check = df.copy()
df_check['mes'] = df_check.ds.dt.month
df_check['outlier_mensual'] = False

for mes in range(1, 13):
    mask = df_check.mes == mes
    vals_mes = df_check.loc[mask, 'y']
    q1, q3 = vals_mes.quantile(0.25), vals_mes.quantile(0.75)
    iqr_m   = q3 - q1
    es_outlier = (vals_mes < q1 - 1.5*iqr_m) | (vals_mes > q3 + 1.5*iqr_m)
    df_check.loc[mask & es_outlier, 'outlier_mensual'] = True

n_out_mes = df_check.outlier_mensual.sum()
print(f"Outliers con IQR por mes: {n_out_mes}")
if n_out_mes:
    print(df_check[df_check.outlier_mensual][['ds','y','mes']])
else:
    print("✅ Sin outliers cuando se aplica IQR por estación")

Outliers con IQR por mes: 8
           ds       y  mes
1  1964-02-29  2672.0    2
7  1964-08-31  2212.0    8
11 1964-12-31  7132.0   12
35 1966-12-31  5534.0   12
42 1967-07-31  3225.0    7
47 1967-12-31  6800.0   12
56 1968-09-30  3521.0    9
83 1970-12-31  5500.0   12


### 3.3 Cambios de Régimen y Anomalías Estructurales

In [9]:
# ── Detección visual: media y varianza móviles ──────────────────────────
ventana = 12
media_movil  = df.y.rolling(ventana, center=True).mean()
std_movil    = df.y.rolling(ventana, center=True).std()

fig = make_subplots(rows=2, cols=1,
    subplot_titles=['Serie + Media Móvil (ventana=12)',
                    'Desv. Estándar Móvil — cambios indican heteroscedasticidad'],
    vertical_spacing=0.15)

fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines', name='Original',
    line=dict(color='lightblue', width=1.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.ds, y=media_movil, mode='lines', name='Media móvil',
    line=dict(color='#2196F3', width=2.5)), row=1, col=1)
fig.add_trace(go.Scatter(x=df.ds, y=std_movil, mode='lines', name='Desv. estándar',
    line=dict(color='#FF5722', width=2), fill='tozeroy',
    fillcolor='rgba(255,87,34,0.1)'), row=2, col=1)

fig.update_layout(height=550, template='plotly_white',
                  title='Detección de Cambios de Régimen — Estadísticas Móviles')
fig.show()

In [10]:
# ── Prueba de Chow (concepto): comparar dos subperíodos ─────────────────
# Dividimos la serie en dos mitades y comparamos sus medias y varianzas
# Una diferencia estadísticamente significativa sugiere cambio de régimen

n_total  = len(df)
mitad    = n_total // 2
y1 = df.y.values[:mitad]
y2 = df.y.values[mitad:]

# Test de Levene: igualdad de varianzas
stat_var, p_var = stats.levene(y1, y2)

# Test t: igualdad de medias
stat_med, p_med = stats.ttest_ind(y1, y2)

print("  TEST DE CAMBIO DE RÉGIMEN — Comparación de dos subperíodos")
print(f"  Período 1: {df.ds.iloc[0].strftime('%Y-%m')} → {df.ds.iloc[mitad-1].strftime('%Y-%m')}")
print(f"  Período 2: {df.ds.iloc[mitad].strftime('%Y-%m')} → {df.ds.iloc[-1].strftime('%Y-%m')}")
print()
print(f"  Media P1={y1.mean():.0f}  |  Media P2={y2.mean():.0f}")
print(f"  Desv. P1={y1.std():.0f}   |  Desv. P2={y2.std():.0f}")
print()
print(f"  Test Levene (varianzas iguales): stat={stat_var:.3f}  p={p_var:.4f}",
      "→ Varianzas distintas ⚠️" if p_var < 0.05 else "→ Varianzas similares ✅")
print(f"  Test t (medias iguales):         stat={stat_med:.3f}  p={p_med:.4f}",
      "→ Medias distintas ⚠️" if p_med < 0.05 else "→ Medias similares ✅")

print()
print("  Interpretación:")
if p_var < 0.05 or p_med < 0.05:
    print("  ⚠️  Hay evidencia de cambio estructural entre períodos.")
    print("     Estrategias: dummy variable, segmentar la serie, o modelar con SARIMA.")
else:
    print("  ✅  Los dos subperíodos son estadísticamente similares.")
    print("     La serie no muestra cambios de régimen significativos.")

  TEST DE CAMBIO DE RÉGIMEN — Comparación de dos subperíodos
  Período 1: 1964-01 → 1968-06
  Período 2: 1968-07 → 1972-12

  Media P1=3313  |  Media P2=3304
  Desv. P1=1317   |  Desv. P2=1275

  Test Levene (varianzas iguales): stat=0.116  p=0.7346 → Varianzas similares ✅
  Test t (medias iguales):         stat=0.037  p=0.9703 → Medias similares ✅

  Interpretación:
  ✅  Los dos subperíodos son estadísticamente similares.
     La serie no muestra cambios de régimen significativos.


### 🔍 Interpretación — Cambios de Régimen

Un **cambio de régimen** ocurre cuando las propiedades estadísticas de la serie cambian abruptamente
(por ejemplo, tras un evento económico, una regulación o un shock externo).

**Estrategias de tratamiento:**

| Estrategia | Cuándo usar |
|-----------|-------------|
| **Variable dummy** | Cambio bien localizado en el tiempo |
| **Segmentar la serie** | Cambio permanente — modelar cada período por separado |
| **Modelo con cambio de punto** | Detección automática (e.g., Prophet, BOCPD) |
| **Remover la tendencia** | Si el "cambio" es en realidad una tendencia no capturada |

> 📌 En esta sesión el diagnóstico es **visual y estadístico básico**. Técnicas más avanzadas
> de detección automática de puntos de cambio (BOCPD, PELT) se verán en semanas posteriores.


---
## 📈 PARTE 4: Análisis de Autocorrelación (ACF)

La ACF mide cuánto se parece la serie a sí misma en distintos instantes del pasado.
Aquí la usamos para confirmar que la serie tiene estructura temporal predecible.

> 📌 Recordatorio de la **Sesión 2:** la ACF de una serie estacionaria decae rápidamente;
> la de una serie no estacionaria decae lentamente. La ACF que veremos aquí también revelará
> el período estacional de la serie.


In [11]:
def plot_acf_plotly(series, title, n_lags=30, color='#2196F3'):
    """
    ACF interactivo con Plotly.
    Barras en rojo = significativas al 95% (fuera del IC).
    """
    arr = np.asarray(series).astype(float)
    arr = arr[~np.isnan(arr)]
    acf_vals = acf(arr, nlags=n_lags, fft=True)
    ci   = 1.96 / np.sqrt(len(arr))
    lags = np.arange(len(acf_vals))

    fig = go.Figure()
    for lag in lags:
        bar_color = color if abs(acf_vals[lag]) <= ci else 'crimson'
        fig.add_trace(go.Scatter(x=[lag, lag], y=[0, acf_vals[lag]], mode='lines',
                                  line=dict(color=bar_color, width=2.5), showlegend=False))
    fig.add_trace(go.Scatter(x=lags, y=acf_vals, mode='markers', showlegend=False,
                              marker=dict(color=[color if abs(v) <= ci else 'crimson'
                                                 for v in acf_vals], size=6)))
    fig.add_hline(y= ci, line_dash='dash', line_color='gray', opacity=0.7,
                  annotation_text=f'IC 95% = ±{ci:.3f}')
    fig.add_hline(y=-ci, line_dash='dash', line_color='gray', opacity=0.7)
    fig.add_hline(y=0,   line_color='black', line_width=0.8)
    fig.update_layout(title=title, xaxis_title='Lag', yaxis_title='Autocorrelación',
                      height=360, template='plotly_white', yaxis=dict(range=[-1.05, 1.05]))
    return fig

fig_acf = plot_acf_plotly(df.y, 'ACF — Ventas Mensuales de Champagne', n_lags=30)
fig_acf.show()

### 🔍 ¿Qué observar en la ACF?

- **Picos en lags 12, 24:** confirman estacionalidad anual (lag estacional = 12 meses)
- **Decaimiento lento:** la ACF no cae a cero rápidamente → la serie NO es estacionaria (confirma lo visto en Sesión 2)
- **Barras rojas:** autocorrelaciones estadísticamente significativas (fuera del intervalo de confianza)

> 📌 **Conexión con la próxima sesión:** en la sesión de ARIMA aprenderemos a usar la PACF
> junto con la ACF para identificar los órdenes p y q de los modelos. Por ahora la ACF
> nos basta para confirmar que hay estructura temporal que un modelo puede aprovechar.


---
## 📐 PARTE 4.5: Tests de Estacionaridad — ADF y KPSS

La ACF mostró decaimiento lento → la serie NO es estacionaria. Los tests **ADF y KPSS**
formalizan esta observación con evidencia estadística.

> ### ¿Por qué hacer esto antes del modelado baseline?
>
> El diagnóstico de estacionaridad forma parte del **análisis exploratorio (EDA)**,
> no del preprocesamiento previo al modelo. La razón:
>
> | Tipo de modelo | ¿Requiere transformar para estacionaridad? | ¿Cómo maneja la no estacionaridad? |
> |---------------|---------------------------------------------|-------------------------------------|
> | **Naive / SeasonalNaive** | ❌ No | La copia mecánicamente |
> | **WindowAverage** | ❌ No | Promedia los valores crudos |
> | **RandomWalkWithDrift** | ❌ No | Fue diseñado para series con drift |
> | **ARIMA / SARIMA** | ✅ Sí | La elimina vía diferenciación (parámetro `d`) |
> | **ML (XGBoost, LightGBM)** | ⚡ Opcional | Features de lag la capturan implícitamente |
> | **DL (RNN, Transformer)** | ⚡ Opcional | Aprenden el patrón directamente |
>
> **Conclusión práctica:** los baselines funcionan sobre datos crudos. Sin embargo,
> el diagnóstico de estacionaridad aquí nos permite:
> - Confirmar qué tipo de serie estamos enfrentando
> - Entender por qué SeasonalNaive funcionará bien y Naive no
> - Preparar el terreno para ARIMA en la próxima sesión, donde sí será obligatorio


In [12]:
# ── Tests ADF y KPSS sobre la serie de Champagne ─────────────────────────
from statsmodels.tsa.stattools import adfuller, kpss

def test_estacionaridad(serie, nombre):
    """
    Aplica ADF + KPSS y entrega la conclusión conjunta.
    Tabla de decisión:
      ADF p<0.05  + KPSS p≥0.05  → ESTACIONARIA
      ADF p≥0.05  + KPSS p<0.05  → NO ESTACIONARIA
      Ambos rechazan / ambos aceptan → INCIERTA
    """
    arr = np.asarray(serie).astype(float)
    arr = arr[~np.isnan(arr)]

    adf_stat, adf_p, _, _, adf_crit, _ = adfuller(arr, autolag='AIC')
    kpss_stat, kpss_p, _, kpss_crit    = kpss(arr, regression='c', nlags='auto')

    adf_est  = adf_p  < 0.05
    kpss_est = kpss_p >= 0.05

    if   adf_est and kpss_est:      conclusion = "✅  ESTACIONARIA"
    elif not adf_est and not kpss_est: conclusion = "❌  NO ESTACIONARIA"
    elif adf_est and not kpss_est:   conclusion = "⚠️  INCIERTA (tendencia?)"
    else:                             conclusion = "⚠️  INCIERTA (cerca del límite)"

    print(f"\n{'═'*58}")
    print(f"  {nombre}")
    print(f"{'═'*58}")
    print(f"  ADF:  stat={adf_stat:8.4f}  p={adf_p:.4f}",
          "→ ES estacionaria ✅" if adf_est else "→ NO estacionaria ❌")
    print(f"  KPSS: stat={kpss_stat:8.4f}  p={kpss_p:.4f}",
          "→ ES estacionaria ✅" if kpss_est else "→ NO estacionaria ❌")
    print(f"  {'─'*54}")
    print(f"  CONCLUSIÓN: {conclusion}")
    return {'adf_p': adf_p, 'kpss_p': kpss_p, 'estacionaria': adf_est and kpss_est}

# Aplicar a la serie original
r_orig = test_estacionaridad(df.y, 'Ventas de Champagne — Serie Original')


══════════════════════════════════════════════════════════
  Ventas de Champagne — Serie Original
══════════════════════════════════════════════════════════
  ADF:  stat= -1.9529  p=0.3076 → NO estacionaria ❌
  KPSS: stat=  0.0229  p=0.1000 → ES estacionaria ✅
  ──────────────────────────────────────────────────────
  CONCLUSIÓN: ⚠️  INCIERTA (cerca del límite)


### 🔍 Interpretación — Tests de Estacionaridad

El código imprimirá el resultado automático. La interpretación general:

- **ADF p > 0.05** → No rechaza raíz unitaria → evidencia de NO estacionaridad ❌
- **KPSS p ≥ 0.05** → No rechaza estacionaridad → evidencia de estacionaridad ✅ *(contradictorio)*
- **Resultado frecuente en esta serie:** caso *incierto* o *cerca del límite*

> 📌 Este es un resultado pedagógicamente valioso:
> **Los tests no siempre coinciden.** La ACF con decaimiento lento y los picos estacionales
> son la evidencia visual más robusta de que la serie tiene estructura no estacionaria.
>
> **Regla práctica:** cuando hay resultados contradictorios, confiar en la evidencia visual
> (ACF, media/std móvil) y diferenciar conservadoramente antes de modelar con ARIMA.
> Para baselines, esto **no cambia nada** — operan sobre los datos crudos.


---
## 🔬 PARTE 5: Test de Ljung-Box

El **Test de Ljung-Box** prueba si múltiples autocorrelaciones son simultáneamente cero.
Nos dice si la serie tiene estructura temporal explotable por un modelo.

$$H_0: \rho_1 = \rho_2 = \cdots = \rho_h = 0 \quad \Rightarrow \quad Q_{LB} = n(n+2)\sum_{k=1}^{h}\frac{\hat{\rho}_k^2}{n-k} \sim \chi^2_h$$


In [13]:
# ── Test de Ljung-Box para múltiples lags ────────────────────────────────
lags_test = [1, 6, 12, 18, 24]

print("TEST DE LJUNG-BOX — Ventas de Champagne")
print("H₀: ρ₁ = ρ₂ = ··· = ρₕ = 0  (no hay autocorrelación)")
print("─" * 62)
print(f"{'Lags':>6}  {'Estadístico Q':>14}  {'p-valor':>10}  {'Conclusión'}")
print("─" * 62)

for lag in lags_test:
    result = acorr_ljungbox(df.y.values, lags=[lag], return_df=True)
    q_stat = result['lb_stat'].iloc[0]
    p_val  = result['lb_pvalue'].iloc[0]
    conc   = "Rechaza H₀ — HAY autocorrelación ❌" if p_val < 0.05              else "No rechaza H₀ — sin autocorrelación ✅"
    print(f"  {lag:>4}  {q_stat:>14.4f}  {p_val:>10.6f}  {conc}")
print("─" * 62)

TEST DE LJUNG-BOX — Ventas de Champagne
H₀: ρ₁ = ρ₂ = ··· = ρₕ = 0  (no hay autocorrelación)
──────────────────────────────────────────────────────────────
  Lags   Estadístico Q     p-valor  Conclusión
──────────────────────────────────────────────────────────────
     1         28.2679    0.000000  Rechaza H₀ — HAY autocorrelación ❌
     6         55.3373    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    12        191.0134    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    18        236.4335    0.000000  Rechaza H₀ — HAY autocorrelación ❌
    24        348.4336    0.000000  Rechaza H₀ — HAY autocorrelación ❌
──────────────────────────────────────────────────────────────


In [14]:
# ── Visualización del p-valor por lag ────────────────────────────────────
lags_all   = list(range(1, 31))
results_lb = acorr_ljungbox(df.y.values, lags=lags_all, return_df=True)
p_valores  = results_lb['lb_pvalue'].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=lags_all, y=p_valores, mode='lines+markers',
                          line=dict(color='#2196F3', width=2),
                          marker=dict(color=['red' if p < 0.05 else 'green'
                                             for p in p_valores], size=7),
                          name='p-valor Ljung-Box'))
fig.add_hline(y=0.05, line_dash='dash', line_color='red',
              annotation_text='α = 0.05', annotation_position='right')

fig.update_layout(
    title='Test de Ljung-Box — p-valores por lag<br>'
          '<sup>Puntos rojos = evidencia de autocorrelación significativa</sup>',
    xaxis_title='Lag', yaxis_title='p-valor',
    height=360, template='plotly_white', yaxis=dict(range=[-0.05, 1.05])
)
fig.show()

### 🔍 Interpretación — Ljung-Box

- **Todos los p-valores son ≈ 0** (muy por debajo de α=0.05): rechazamos H₀ para todos los lags
- La serie **tiene autocorrelación significativa** en múltiples horizontes
- Conclusión: la serie NO es ruido blanco; **hay patrones predecibles** que un modelo puede explotar
- Esto justifica ir más allá de inspección visual y construir modelos formales

> 🔑 El test de Ljung-Box es la misma herramienta que usaremos en cada semana para
> **diagnosticar residuos**: si los residuos de un modelo pasan el test (p ≥ 0.05),
> el modelo capturó toda la estructura temporal disponible.


---
## ✂️ PARTE 6: Train-Test Split Temporal

**Regla crítica:** NUNCA aleatorizar. El tiempo tiene dirección y los modelos de series
temporales aprenden del pasado para predecir el futuro.


In [15]:
# ── Split temporal 80/20 ────────────────────────────────────────────────
fecha_corte = '1971-01-01'
train = df[df.ds < fecha_corte].copy()
test  = df[df.ds >= fecha_corte].copy()

print(f"TRAIN: {len(train)} observaciones  ({train.ds.min().strftime('%Y-%m')} → {train.ds.max().strftime('%Y-%m')})")
print(f"TEST:  {len(test)}  observaciones  ({test.ds.min().strftime('%Y-%m')} → {test.ds.max().strftime('%Y-%m')})")
print(f"Proporción: {len(train)/len(df)*100:.0f}% / {len(test)/len(df)*100:.0f}%")

# Visualización del split
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.ds, y=train.y, mode='lines+markers',
                          name='Train', line=dict(color='#2196F3', width=2),
                          marker=dict(size=3)))
fig.add_trace(go.Scatter(x=test.ds, y=test.y, mode='lines+markers',
                          name='Test', line=dict(color='#FF5722', width=2),
                          marker=dict(size=3)))
# add_shape evita el bug de pandas 2.0+: add_vline falla con fechas
fig.add_shape(type='line', xref='x', yref='paper',
              x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
              line=dict(dash='dash', color='gray', width=1.5))
fig.add_annotation(x=fecha_corte, y=1.02, yref='paper',
                   text='Corte', showarrow=False,
                   font=dict(size=10, color='gray'), xanchor='left')
fig.update_layout(title='Train-Test Split Temporal — 80/20',
                  xaxis_title='Fecha', yaxis_title='Ventas',
                  height=380, template='plotly_white')
fig.show()

TRAIN: 84 observaciones  (1964-01 → 1970-12)
TEST:  24  observaciones  (1971-01 → 1972-12)
Proporción: 78% / 22%


---
## 🤖 PARTE 7: Baselines con `statsforecast`

Implementamos los 4 modelos baseline vistos en clase usando `StatsForecast`.
El patrón es siempre el mismo: `fit(train)` → `predict(h=horizonte)`.

> Este es el **patrón estándar de Nixtla** que usaremos con ARIMA, ML y DL en las semanas siguientes.


In [16]:
# ── Instanciar y ajustar todos los modelos en una sola llamada ──────────
h = len(test)   # horizonte = longitud del conjunto de prueba

sf = StatsForecast(
    models=[
        Naive(),                        # ŷ = y_t (último valor)
        SeasonalNaive(season_length=12), # ŷ = y_{t-12} (mismo mes año anterior)
        WindowAverage(window_size=3),    # ŷ = promedio últimas 3 obs.
        RandomWalkWithDrift()            # ŷ = y_t + drift (tendencia histórica)
    ],
    freq='ME'    # frecuencia mensual
)

sf.fit(train)
preds = sf.predict(h=h)

print("Pronósticos generados:")
print(f"  Modelos: {[c for c in preds.columns if c not in ['unique_id','ds']]}")
print(f"  Horizonte: {h} meses")
preds.head()

Pronósticos generados:
  Modelos: ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']
  Horizonte: 24 meses


,unique_id,ds,Naive,SeasonalNaive,WindowAverage,RWD
0,Champagne,1971-01-31,5500.0,1701.0,4827.333333,5532.349398
1,Champagne,1971-02-28,5500.0,1715.0,4827.333333,5564.698795
2,Champagne,1971-03-31,5500.0,2360.0,4827.333333,5597.048193
3,Champagne,1971-04-30,5500.0,2560.0,4827.333333,5629.397590
4,Champagne,1971-05-31,5500.0,2838.0,4827.333333,5661.746988


In [17]:
# ── Visualización: pronósticos vs valores reales ─────────────────────────
test_preds = test.merge(preds, on=['unique_id', 'ds'])
modelos    = ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']
colores    = ['#FF5722', '#4CAF50', '#9C27B0', '#FF9800']

fig = go.Figure()

# Datos históricos completos (fondo)
fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines', name='Histórico',
                          line=dict(color='lightgray', width=1.5)))

# Período de prueba real
fig.add_trace(go.Scatter(x=test.ds, y=test.y, mode='lines+markers',
                          name='Real (test)', line=dict(color='black', width=2.5),
                          marker=dict(size=5)))

# Pronósticos de cada modelo
for modelo, color in zip(modelos, colores):
    fig.add_trace(go.Scatter(x=test_preds.ds, y=test_preds[modelo],
                              mode='lines+markers', name=modelo,
                              line=dict(color=color, width=2, dash='dot'),
                              marker=dict(size=4)))

fig.add_shape(type='line', xref='x', yref='paper',
              x0=fecha_corte, x1=fecha_corte, y0=0, y1=1,
              line=dict(dash='dash', color='gray', width=1.5))
fig.add_annotation(x=fecha_corte, y=1.02, yref='paper',
                   text='Inicio test', showarrow=False,
                   font=dict(size=10, color='gray'), xanchor='left')
fig.update_layout(
    title='Comparación de Baselines — Champagne Sales (1964–1972)',
    xaxis_title='Fecha', yaxis_title='Ventas',
    height=480, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

### 🔍 ¿Qué observar en los pronósticos?
- **Naive:** predice el último valor observado — una línea plana. Ignora toda la estacionalidad.
- **SeasonalNaive:** replica el mismo mes del año anterior — capta el patrón estacional perfectamente.
- **WindowAverage:** promedio de las últimas 3 observaciones — suaviza pero pierde la estacionalidad.
- **RWD (Random Walk with Drift):** extrapola la tendencia histórica — asciende linealmente.

La inspección visual ya sugiere al ganador. Lo confirmamos con métricas formales.


---
## 📊 PARTE 8: Métricas de Evaluación

Cuantificamos el error de cada modelo con las métricas vistas en clase.

| Métrica | Fórmula conceptual | Característica |
|---------|-------------------|---------------|
| **MAE** | Media de `\|error\|` | Robusta, misma unidad que los datos |
| **RMSE** | Raíz de la media de `error²` | Penaliza errores grandes |
| **sMAPE** | Error porcentual simétrico | Comparable entre series distintas |


In [18]:
# ── Calcular métricas con utilsforecast ──────────────────────────────────
modelos_col = ['Naive', 'SeasonalNaive', 'WindowAverage', 'RWD']

# MASE requiere la serie de entrenamiento como denominador (MAE del Naive sobre train)
naive_train_mae = float(np.abs(np.diff(train.y.values)).mean())

evaluacion = evaluate(
    test_preds,
    metrics=[mae, rmse, smape],
    models=modelos_col,
    target_col='y'
)

# MASE = MAE_test / MAE_naive_train
fila_mae = evaluacion[evaluacion.metric == 'mae'].copy()
nueva_fila_mase = fila_mae.copy()
nueva_fila_mase['metric'] = 'mase'

for m in modelos_col:
    nueva_fila_mase[m] = nueva_fila_mase[m] / naive_train_mae

# Concatenar a la tabla original
evaluacion = pd.concat([evaluacion, nueva_fila_mase], ignore_index=True)

print(f"\nMAE del Naive sobre train (denominador MASE): {naive_train_mae:.2f}")

# Reformatear para mejor visualización
print("=" * 60)
print("  RESULTADOS POR MÉTRICA — MODELOS BASELINE")
print("=" * 60)
display(evaluacion)

print("\n📌 Mejor modelo por métrica:")
for metrica in ['mae', 'rmse', 'smape', 'mase']:
    fila = evaluacion[evaluacion.metric == metrica]
    mejor = fila[modelos_col].values.argmin()
    nombre = modelos_col[mejor]
    valor  = fila[modelos_col].values[0][mejor]
    print(f"   {metrica.upper():>6}: {nombre}  ({valor:.4f})")


MAE del Naive sobre train (denominador MASE): 761.63
  RESULTADOS POR MÉTRICA — MODELOS BASELINE


,unique_id,metric,Naive,SeasonalNaive,WindowAverage,RWD
0,Champagne,mae,2353.875000,313.708333,1875.430556,2662.643072
1,Champagne,rmse,2568.215828,380.488995,2006.763269,2899.517981
2,Champagne,smape,0.288226,0.054245,0.247558,0.311593
3,Champagne,mase,3.090590,0.411893,2.462402,3.495996



📌 Mejor modelo por métrica:
      MAE: SeasonalNaive  (313.7083)
     RMSE: SeasonalNaive  (380.4890)
    SMAPE: SeasonalNaive  (0.0542)
     MASE: SeasonalNaive  (0.4119)


In [19]:
# ── Visualización comparativa de métricas ────────────────────────────────
fig = make_subplots(rows=1, cols=3, subplot_titles=['MAE', 'RMSE', 'sMAPE'])
colores_m = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']

for col_idx, metrica in enumerate(['mae', 'rmse', 'smape'], start=1):
    fila = evaluacion[evaluacion.metric == metrica]
    vals = [float(fila[m].values[0]) for m in modelos_col]
    mejor_idx = int(np.argmin(vals))

    bar_colors = ['gold' if i == mejor_idx else colores_m[i]
                  for i in range(len(modelos_col))]
    fig.add_trace(
        go.Bar(x=modelos_col, y=vals, marker_color=bar_colors,
               name=metrica, showlegend=False,
               text=[f'{v:.4f}' if metrica == 'smape' else f'{v:.1f}'
                     for v in vals], textposition='outside'),
        row=1, col=col_idx
    )

fig.update_layout(height=400, template='plotly_white',
                  title='Comparación de Métricas por Modelo<br>'
                        '<sup>Barra dorada = mejor modelo en esa métrica</sup>')
fig.show()

### 🔍 Conclusión de métricas

**SeasonalNaive** debería ser el ganador claro en las tres métricas — y por un margen amplio.
Esto se explica porque la serie tiene estacionalidad fuerte y estable: el mejor predictor
para diciembre de 1971 es diciembre de 1970.

**Lección clave:** un modelo complejo (ARIMA, ML, DL) solo tiene valor si supera a este
baseline. Si tu modelo no supera al SeasonalNaive en esta serie, algo está mal.


---
## 🔬 Cierre del Ciclo: Ljung-Box sobre Residuos del Mejor Modelo

El test de Ljung-Box tiene **dos usos** (como vimos en los slides):
1. **Serie cruda:** confirma que hay estructura predecible → conviene modelar
2. **Residuos del modelo:** verifica si quedaron patrones sin capturar → ¿es ruido blanco?

Si los residuos pasan el test (p ≥ 0.05), el modelo capturó todo lo disponible.
Si no, queda estructura que otro modelo (como ARIMA) podría aprovechar.


In [20]:
# ── Residuos del SeasonalNaive (mejor modelo) ────────────────────────────
residuos     = test_preds['y'] - test_preds['SeasonalNaive']
residuos_arr = residuos.values

print(f"Residuos — media: {residuos_arr.mean():.2f}  |  std: {residuos_arr.std():.2f}")
print(f"Esperado en ruido blanco: media ≈ 0, desviación estándar constante")

# ── Ljung-Box sobre los residuos ─────────────────────────────────────────
print("\nLJUNG-BOX SOBRE RESIDUOS DEL SeasonalNaive:")
print("─" * 58)
for lag in [1, 6, 12]:
    result = acorr_ljungbox(residuos_arr, lags=[lag], return_df=True)
    p_val  = result['lb_pvalue'].iloc[0]
    conc = "Quedan patrones ⚠️  → margen de mejora" if p_val < 0.05  else "Residuos ≈ ruido blanco ✅"
    print(f"  Lag {lag:>2}: p={p_val:.4f}  →  {conc}")

# ── ACF de los residuos (usa el acf importado en la celda de setup) ──────
acf_resid = acf(residuos_arr, nlags=12, fft=True)
ci_r = 1.96 / np.sqrt(len(residuos_arr))

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Residuos en el tiempo', 'ACF de Residuos'])

fig.add_trace(go.Scatter(x=test_preds.ds, y=residuos, mode='lines+markers',
    line=dict(color='#FF5722', width=2), name='Residuos'), row=1, col=1)
fig.add_hline(y=0,                     line_dash='dash', line_color='black',   row=1, col=1)
fig.add_hline(y= 2*residuos_arr.std(), line_dash='dot',  line_color='gray',    row=1, col=1)
fig.add_hline(y=-2*residuos_arr.std(), line_dash='dot',  line_color='gray',    row=1, col=1)

for lag_r in range(len(acf_resid)):
    bc = 'crimson' if abs(acf_resid[lag_r]) > ci_r else '#4CAF50'
    fig.add_trace(go.Scatter(x=[lag_r, lag_r], y=[0, acf_resid[lag_r]],
        mode='lines', line=dict(color=bc, width=3), showlegend=False), row=1, col=2)
fig.add_hline(y= ci_r, line_dash='dash', line_color='gray', row=1, col=2)
fig.add_hline(y=-ci_r, line_dash='dash', line_color='gray', row=1, col=2)

fig.update_layout(height=380, template='plotly_white', showlegend=False,
    title='Diagnóstico de Residuos — SeasonalNaive<br>'
          '<sup>Verde = no significativo (ruido) | Rojo = patrón remanente</sup>')
fig.show()

Residuos — media: 224.38  |  std: 307.29
Esperado en ruido blanco: media ≈ 0, desviación estándar constante

LJUNG-BOX SOBRE RESIDUOS DEL SeasonalNaive:
──────────────────────────────────────────────────────────
  Lag  1: p=0.0534  →  Residuos ≈ ruido blanco ✅
  Lag  6: p=0.0071  →  Quedan patrones ⚠️  → margen de mejora
  Lag 12: p=0.0017  →  Quedan patrones ⚠️  → margen de mejora


---
## 🔄 PARTE 9: Time Series Cross-Validation

El split simple de train/test tiene alta varianza — depende de qué período específico
cayó en el test. La **validación cruzada temporal** genera múltiples ventanas de evaluación,
produciendo métricas más confiables.


In [21]:
# ── Cross-Validation con StatsForecast ───────────────────────────────────
# n_windows=3: 3 folds de validación
# h=12:        horizonte de 12 meses en cada fold
# step_size=12: los folds se desplazan de año en año

cv_results = sf.cross_validation(
    df=df,
    h=12,
    n_windows=3,
    step_size=12
)

print(f"Resultados CV: {cv_results.shape[0]} filas × {cv_results.shape[1]} columnas")
print(f"Folds (cutoffs): {cv_results.cutoff.unique().tolist()}")
cv_results.head(6)

Resultados CV: 36 filas × 8 columnas
Folds (cutoffs): [Timestamp('1969-12-31 00:00:00'), Timestamp('1970-12-31 00:00:00'), Timestamp('1971-12-31 00:00:00')]


,unique_id,ds,cutoff,y,Naive,SeasonalNaive,WindowAverage,RWD
0,Champagne,1970-01-31,1969-12-31,1701.0,6087.0,1945.0,5236.0,6133.084507
1,Champagne,1970-02-28,1969-12-31,1715.0,6087.0,2046.0,5236.0,6179.169014
2,Champagne,1970-03-31,1969-12-31,2360.0,6087.0,2318.0,5236.0,6225.253521
3,Champagne,1970-04-30,1969-12-31,2560.0,6087.0,2685.0,5236.0,6271.338028
4,Champagne,1970-05-31,1969-12-31,2838.0,6087.0,2731.0,5236.0,6317.422535
5,Champagne,1970-06-30,1969-12-31,2640.0,6087.0,3010.0,5236.0,6363.507042


In [22]:
# ── Métricas promedio por fold ────────────────────────────────────────────
# cv_results ya tiene columna 'y' — sin necesidad de rename
# Verificar columnas disponibles antes de evaluar
cv_models = [c for c in modelos_col if c in cv_results.columns]
cv_eval = evaluate(
    cv_results,
    metrics=[mae, rmse, smape],
    models=cv_models,
    target_col='y'
)

print("Métricas promedio en Cross-Validation (3 folds):")
print(cv_eval.to_string(index=False))

# ── Visualización de predicciones por fold ────────────────────────────────
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.ds, y=df.y, mode='lines',
                          name='Serie real', line=dict(color='black', width=2)))

colores_cv = ['#FF5722', '#4CAF50', '#9C27B0']
for i, cutoff in enumerate(sorted(cv_results.cutoff.unique())):
    fold = cv_results[cv_results.cutoff == cutoff]
    # Solo SeasonalNaive (mejor modelo)
    fig.add_trace(go.Scatter(
        x=fold.ds, y=fold.SeasonalNaive, mode='lines+markers',
        name=f'SeasonalNaive Fold {i+1} (cutoff {cutoff.strftime("%Y-%m")})',
        line=dict(color=colores_cv[i], width=2, dash='dot'),
        marker=dict(size=5)
    ))
    cutoff_str = cutoff.strftime('%Y-%m-%d')
    fig.add_shape(type='line', xref='x', yref='paper',
                  x0=cutoff_str, x1=cutoff_str, y0=0, y1=1,
                  line=dict(dash='dash', color=colores_cv[i], width=1),
                  opacity=0.4)

fig.update_layout(
    title='Time Series Cross-Validation — SeasonalNaive (3 folds)',
    xaxis_title='Fecha', yaxis_title='Ventas',
    height=420, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

Métricas promedio en Cross-Validation (3 folds):
unique_id     cutoff metric       Naive  SeasonalNaive  WindowAverage         RWD
Champagne 1969-12-31    mae 3057.000000     230.833333    2250.000000 3356.549296
Champagne 1970-12-31    mae 2342.666667     281.500000    1894.222222 2450.738956
Champagne 1971-12-31    mae 2816.083333     281.250000    2286.472222 3036.741228
Champagne 1969-12-31   rmse 3264.934941     281.983156    2486.159119 3504.795478
Champagne 1970-12-31   rmse 2572.223066     351.805581    2022.047079 2712.037279
Champagne 1971-12-31   rmse 3053.670510     342.256605    2480.848908 3226.280788
Champagne 1969-12-31  smape    0.354566       0.037614       0.293069    0.375820
Champagne 1970-12-31  smape    0.287595       0.043926       0.249799    0.295864
Champagne 1971-12-31  smape    0.322652       0.046710       0.282506    0.338743


### 🔍 Interpretación — Cross-Validation

- Cada fold usa un período diferente de entrenamiento y prueba
- Las métricas promedio a través de los 3 folds son **más robustas** que una sola evaluación
- Si el modelo es consistente, sus métricas deberían ser similares en todos los folds
- Si hay mucha varianza entre folds, puede indicar inestabilidad o cambios en la serie

> 🔑 `StatsForecast.cross_validation()` maneja automáticamente el manejo temporal
> — no hay riesgo de data leakage.


---
## 🎯 PARTE 10: Práctica Guiada

Aplica el pipeline completo sobre el mismo dataset pero con un análisis más profundo.

### Ejercicio: Diagnóstico completo + Comparación avanzada

**Instrucciones:**

1. **Calidad de datos:** aplica IQR por mes sobre el dataset de champagne.
   ¿Encuentra outliers reales o todos son variaciones estacionales legítimas?

2. **ACF estacional:** ¿en qué lags exactos son significativas las autocorrelaciones?
   Relaciona esos lags con el ciclo de ventas de champagne.

3. **Ljung-Box sobre los residuos del SeasonalNaive:**
   Calcula los residuos `e_t = y_t - ŷ_t` del SeasonalNaive en el test
   y aplica Ljung-Box. ¿Son ruido blanco o quedan patrones?

4. **Experimentar con ventana del WindowAverage:**
   Prueba `window_size` = 2, 3, 6, 12. ¿Cuál minimiza el MAE?

5. **Reflexión:** dado que SeasonalNaive ya captura bien la estacionalidad,
   ¿qué necesita capturar adicionalmente un modelo ARIMA para mejorar?


In [23]:
# ── EJERCICIO 1: IQR por mes ─────────────────────────────────────────────
# TU CÓDIGO AQUÍ
# df_check = df.copy()
# df_check['mes'] = df_check.ds.dt.month
# ... IQR por mes ...


In [24]:
# ── EJERCICIO 3: Ljung-Box sobre residuos del SeasonalNaive ──────────────
# TU CÓDIGO AQUÍ
# residuos = test_preds['y'] - test_preds['SeasonalNaive']
# result_resid = acorr_ljungbox(residuos, lags=[1, 6, 12], return_df=True)
# print(result_resid)
# Interpretación: ¿quedan patrones?


In [25]:
# ── EJERCICIO 4: Optimizar window_size ───────────────────────────────────
# TU CÓDIGO AQUÍ
# best_k, best_mae = None, float('inf')
# for k in [2, 3, 6, 12]:
#     sf_k = StatsForecast(models=[WindowAverage(window_size=k)], freq='ME')
#     sf_k.fit(train)
#     p_k = sf_k.predict(h=h)
#     # calcular MAE y actualizar mejor...


---
## ✅ Resumen de la Sesión

| Tema | Herramienta | Función clave |
|------|-------------|---------------|
| Valores faltantes | pandas | `.ffill()`, `.interpolate()`, media estacional |
| Outliers | scipy | `stats.iqr()`, `stats.zscore()` |
| Cambios de régimen | scipy | `stats.levene()`, `stats.ttest_ind()` |
| ACF | statsmodels + Plotly | `acf()` + gráfico interactivo |
| Tests ADF + KPSS | statsmodels | `adfuller()`, `kpss()` — diagnóstico EDA |
| Ljung-Box (serie + residuos) | statsmodels | `acorr_ljungbox()` |
| Train-Test split | pandas | Filtro temporal — nunca aleatorio |
| Baselines | statsforecast | `Naive`, `SeasonalNaive`, `WindowAverage`, `RandomWalkWithDrift` |
| Métricas | utilsforecast | `mae`, `rmse`, `smape`, `evaluate()` |
| Cross-Validation | statsforecast | `sf.cross_validation()` |